# Gold 3_00 — Staff Emails (Who Is Staff)

One row per **staff person**, extracted from the identity spine (`silver.user_identity_map`, 2_8).
This is the single source of truth for which emails belong to staff — 3_12 (`lead_emails`)
subtracts `all_emails` to produce the client email view.

## The staff rule — whole-person, any marker suffices

A person (spine master) is staff if ANY of:

1. **Organizer login** — carries a 48-char CRM-login hash (`attendees.user_id`). Only a logged-in staff member can organize an activity; a client can never produce one.
2. **Company domain** — any real email on `@dataengineeracademy.com`.
3. **Registered** — any real email registered in `close_crm_users` (the CRM's own staff list).

Once staff, **the entire person is staff**: every hash, real email, and masked email.
A staff member accumulates client markers naturally (masked contact records, attendee rows,
Calendly bookings) — that does not make them a client. The inverse never happens: a client
never carries an organizer login. Staff-first classification, client = NOT in staff.

Known residual: masked correspondence addresses that carry NO real email and NO organizer
login (e.g. shared masked senders spanning many leads) remain classified as non-staff by
construction — they cannot be caught by any current rule.

In [0]:
# ============================================================================
# BUILD - crm_ingestion.gold.staff_emails
# ============================================================================
# One row per STAFF person. Whole-master rule: one staff marker makes the ENTIRE
# person staff - every hash, real email, masked email. Runs after 2_8 (reads the
# spine). 3_12 subtracts all_emails to produce the client email view; the spine
# remains the everyone-lookup.

spark.sql("""
CREATE OR REPLACE TABLE crm_ingestion.gold.staff_emails AS
WITH registered AS (
    SELECT DISTINCT LOWER(TRIM(email)) AS email
    FROM crm_ingestion.silver.close_crm_users_processed
    WHERE email IS NOT NULL
),
flagged AS (
    SELECT
        sp.master_user_id,
        sp.user_32digits,
        sp.real_emails,
        sp.masked_emails,
        sp.lead_ids,
        sp.cont_ids,
        sp.source_tables,
        (size(filter(sp.user_32digits, h -> LENGTH(h) = 48)) > 0) AS is_organizer_login,
        (size(filter(sp.real_emails, e -> LOWER(e) LIKE '%@dataengineeracademy.com')) > 0) AS is_dea_domain,
        EXISTS (
            SELECT 1 FROM registered r WHERE array_contains(sp.real_emails, r.email)
        ) AS is_registered
    FROM crm_ingestion.silver.user_identity_map sp
)
SELECT
    master_user_id,
    is_organizer_login,
    is_dea_domain,
    is_registered,
    user_32digits,
    real_emails,
    masked_emails,
    array_distinct(concat(real_emails, masked_emails)) AS all_emails,
    lead_ids,
    cont_ids,
    source_tables
FROM flagged
WHERE is_organizer_login OR is_dea_domain OR is_registered
""")

# ------------------------------------------------------------------------
# Column comments: the rule in the data dictionary, where it belongs.
# ------------------------------------------------------------------------
column_comments = {}
column_comments["is_organizer_login"] = "Person carries a 48-char CRM-login hash (attendees.user_id) - they organized an activity. Only staff can produce this marker; a client never can."
column_comments["is_dea_domain"] = "Person carries a real email on the company domain (@dataengineeracademy.com)."
column_comments["is_registered"] = "Person carries an email registered in close_crm_users - the CRM staff list."
column_comments["user_32digits"] = "All identity hashes of this staff person (48-char login ids and 32-hex masked person hashes)."
column_comments["real_emails"] = "All real (unmasked) emails of this staff person - personal and company."
column_comments["masked_emails"] = "All masked correspondence emails of this staff person."
column_comments["all_emails"] = "real_emails + masked_emails - the complete staff email blocklist. 3_12 lead_emails subtracts exactly this set."
column_comments["lead_ids"] = "Leads this staff person ever corresponded on or was a contact of - provenance only."

for col in column_comments:
    spark.sql(
        "ALTER TABLE crm_ingestion.gold.staff_emails ALTER COLUMN " + col
        + " COMMENT '" + column_comments[col].replace("'", "") + "'"
    )

spark.sql(
    "COMMENT ON TABLE crm_ingestion.gold.staff_emails IS "
    "'One row per staff person (whole-master rule: any one marker - 48-char organizer login hash, "
    "@dataengineeracademy.com email, or close_crm registration - makes the entire person staff). "
    "The single staff email blocklist: 3_12 lead_emails subtracts all_emails to produce the client "
    "email view; the spine silver.user_identity_map remains the everyone-lookup.'"
)

result = spark.table("crm_ingestion.gold.staff_emails")
print("staff_emails built: {:,} staff persons".format(result.count()))
result.groupBy("is_organizer_login", "is_dea_domain", "is_registered").count().orderBy("count", ascending=False).show()

In [0]:
%sql
-- Validation 1: detection breakdown - which markers fire, how many staff, and how
-- many lead links each combination carries.
SELECT
    is_organizer_login, is_dea_domain, is_registered,
    COUNT(*) AS staff_persons,
    SUM(size(lead_ids)) AS lead_links,
    SUM(size(all_emails)) AS email_addresses
FROM crm_ingestion.gold.staff_emails
GROUP BY 1, 2, 3
ORDER BY staff_persons DESC

In [0]:
%sql
-- Validation 2: the busiest staff - who touches the most leads. These are the
-- people whose emails were polluting customer_emails the most before the split.
SELECT
    master_user_id,
    real_emails,
    size(masked_emails) AS n_masked,
    size(lead_ids) AS n_leads,
    source_tables
FROM crm_ingestion.gold.staff_emails
ORDER BY n_leads DESC
LIMIT 15

In [0]:
# Validation 3: known faces.
from pyspark.sql.functions import expr
# Afnan and Monisha must be here with ALL their emails (gmail, @dea, masked);
# the owner's personal yahoo address (client-side markers only, no organizer
# login, no company email) must NOT be.
known = spark.table('crm_ingestion.gold.staff_emails') \
    .filter("array_contains(all_emails, 'afnan.bvs@gmail.com') OR array_contains(all_emails, 'monisha@dataengineeracademy.com')") \
    .select('master_user_id', 'is_organizer_login', 'is_dea_domain', 'is_registered', 'real_emails', 'masked_emails', expr('size(lead_ids)').alias('n_leads'))
known.show(truncate=False)

n_yahoo = spark.sql("SELECT COUNT(*) FROM crm_ingestion.gold.staff_emails WHERE array_contains(all_emails, 'scottsbv@yahoo.com')").collect()[0][0]
print(f"scottsbv@yahoo.com in staff table (must be 0): {n_yahoo}")

n_staff_emails = spark.sql("SELECT COUNT(DISTINCT e) FROM crm_ingestion.gold.staff_emails LATERAL VIEW explode(all_emails) e AS e").collect()[0][0]
print(f"distinct staff email addresses on the blocklist: {n_staff_emails}")